In [1]:
!pip install langchain langchain-groq gradio

In [2]:
import os
import json
import gradio as gr

from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser

In [3]:
os.environ["GROQ_API_KEY"] = "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"

In [8]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.7         ## paramiter hai batata hai ki AI answer kitana creative ya random honga
)

In [9]:
movie_prompt = ChatPromptTemplate.from_template("""                    
You are a movie expert.

Analyze the following movie:

Movie Name: {movie_name}

Return ONLY valid JSON in this format:

{{
    "title": "movie name",
    "genre": ["genre1", "genre2"],
    "year": "release year",
    "themes": ["theme1", "theme2"],
    "mood": "movie mood",
    "summary": "short movie summary"
}}
""")

In [10]:
parser = JsonOutputParser()

In [11]:
movie_chain = movie_prompt | llm | parser

In [12]:
recommendation_prompt = ChatPromptTemplate.from_template("""
You are a movie recommendation expert.

Based on this movie information:

{movie_analysis}

Recommend 3 similar movies.

Return ONLY valid JSON:

{{
    "viewer_type": "What type of viewer would enjoy this movie",
    
    "recommendations": [
        {{
            "title": "Movie Name",
            "reason": "Why this movie is recommended"
        }},
        {{
            "title": "Movie Name",
            "reason": "Why this movie is recommended"
        }},
        {{
            "title": "Movie Name",
            "reason": "Why this movie is recommended"
        }}
    ]
}}
""")

In [13]:
recommendation_chain = recommendation_prompt | llm | parser

In [14]:
def analyze_movie(movie_name):

    # Chain 1
    movie_analysis = movie_chain.invoke(                      ##  invoke AI model ko call karne  aur response lene ke liye use hota hai
        {
            "movie_name": movie_name
        }
    )

    # Chain 2
    recommendations = recommendation_chain.invoke(
        {
            "movie_analysis": json.dumps(movie_analysis)               ##  dumps python dictionary ko jeson string mai convert karne ke liye
        }
    )

    # Final Result
    final_result = {
        "movie": movie_name,
        "analysis": movie_analysis,
        "recommendations": recommendations
    }

    return json.dumps(final_result, indent=4)                      ## 4 spaces ke liye simple output ke spaces ko handle karne ke  liye ye use hota hai

In [15]:
result = analyze_movie("Interstellar")

print(result)

{
    "movie": "Interstellar",
    "analysis": {
        "title": "Interstellar",
        "genre": [
            "Science Fiction",
            "Adventure",
            "Drama"
        ],
        "year": "2014",
        "themes": [
            "Love",
            "Time Dilation",
            "Survival",
            "Sacrifice",
            "Exploration",
            "Family"
        ],
        "mood": "Contemplative and Hopeful",
        "summary": "In a near-future Earth facing ecological collapse, a former pilot joins a daring interstellar mission through a wormhole to find a new habitable world, confronting relativistic time, black holes, and the power of love across dimensions to secure humanity's future."
    },
    "recommendations": {
        "viewer_type": "Sci\u2011fi enthusiasts who enjoy thoughtful, emotionally resonant space epics that blend scientific concepts with human drama",
        "recommendations": [
            {
                "title": "Gravity",
                

In [16]:
def movie_ai(movie_name):

    if movie_name.strip() == "":                        ## strip string ke starting or ending ke extra spaces ko nikalne ke liye use hota hai
        return "Please enter a movie name!"

    return analyze_movie(movie_name)

In [19]:
# -------------------------
# BEAUTIFUL GRADIO UI
# -------------------------

with gr.Blocks(
    title="Movie AI Assistant"
) as demo:

    # HEADER
    gr.Markdown("""
    # 🎬 Movie AI Assistant
    
    ### Discover • Analyze • Recommend
    
    Enter your favorite movie and let AI analyze it
    and recommend similar movies for you! 🍿
    """)


    with gr.Row():

        # LEFT SIDE
        with gr.Column(scale=1):

            gr.Markdown("## 🎥 Enter Movie")

            movie_input = gr.Textbox(
                label="Movie Name",
                placeholder="Example: Interstellar",
                lines=1
            )

            analyze_button = gr.Button(
                "✨ Analyze Movie",
                variant="primary",
                size="lg"
            )


            gr.Markdown("""
            ### 💡 Try These Movies
            
            🎬 Interstellar  
            🎬 Inception  
            🎬 Titanic  
            🎬 Avatar  
            🎬 The Dark Knight
            """)


        # RIGHT SIDE
        with gr.Column(scale=2):

            gr.Markdown("## 🤖 AI Analysis")

            output = gr.Textbox(
                label="Movie Analysis & Recommendations",
                lines=25,
                max_lines=30
            )


    # EXAMPLES

    gr.Examples(
        examples=[
            ["Interstellar"],
            ["Inception"],
            ["Avatar"],
            ["Titanic"],
            ["The Dark Knight"]
        ],

        inputs=movie_input
    )


    # BUTTON ACTION

    analyze_button.click(
        fn=analyze_movie,
        inputs=movie_input,
        outputs=output
    )


# RUN APP

demo.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


In [18]:
demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [20]:
# -------------------------
# BEAUTIFUL DARK GRADIO UI
# -------------------------

custom_css = """
body {
    background: #0b0b0f;
}

.gradio-container {
    background: #0b0b0f !important;
    color: white !important;
}

/* Main Title */
.title {
    text-align: center;
    font-size: 42px;
    font-weight: bold;
    color: #ffffff;
    margin-bottom: 5px;
}

.subtitle {
    text-align: center;
    color: #a1a1aa;
    font-size: 18px;
    margin-bottom: 30px;
}

/* Cards */
.card {
    background: #18181f;
    border: 1px solid #2a2a35;
    border-radius: 18px;
    padding: 25px;
}

/* Movie Button */
.analyze-btn {
    height: 50px;
    font-size: 18px !important;
    font-weight: bold !important;
    border-radius: 12px !important;
}

/* Output */
.output-box textarea {
    background: #111118 !important;
    color: white !important;
    border-radius: 12px !important;
}

/* Footer */
.footer {
    text-align: center;
    color: #71717a;
    margin-top: 30px;
}
"""


with gr.Blocks(
    css=custom_css,
    theme=gr.themes.Base()
) as demo:

    # -------------------------
    # HEADER
    # -------------------------

    gr.HTML("""
    <div class="title">
        🎬 Movie AI Assistant
    </div>

    <div class="subtitle">
        Discover • Analyze • Recommend 🍿
    </div>
    """)


    # -------------------------
    # MAIN SECTION
    # -------------------------

    with gr.Row():

        # LEFT COLUMN

        with gr.Column(scale=1):

            with gr.Group():

                gr.Markdown("""
                ## 🎥 Find Your Movie

                Enter any movie name and let our AI
                analyze it for you.
                """)

                movie_input = gr.Textbox(
                    label="🎬 Movie Name",
                    placeholder="Try: Interstellar",
                    lines=1
                )

                analyze_button = gr.Button(
                    "✨ Analyze Movie",
                    variant="primary",
                    elem_classes="analyze-btn"
                )


                gr.Markdown("""
                ### 🍿 Popular Movies

                Try these examples:

                🎬 Interstellar

                🎬 Inception

                🎬 Avatar

                🎬 Titanic

                🎬 The Dark Knight
                """)


        # RIGHT COLUMN

        with gr.Column(scale=2):

            with gr.Group():

                gr.Markdown("""
                ## 🤖 AI Movie Analysis

                Your personalized movie analysis and
                recommendations will appear here.
                """)

                output = gr.Textbox(
                    label="🎬 Results",
                    placeholder="Your AI movie analysis will appear here...",
                    lines=25,
                    max_lines=30,
                    elem_classes="output-box"
                )


    # -------------------------
    # EXAMPLE MOVIES
    # -------------------------

    gr.Markdown("## 🎞️ Quick Examples")

    gr.Examples(
        examples=[
            ["Interstellar"],
            ["Inception"],
            ["Avatar"],
            ["Titanic"],
            ["The Dark Knight"],
            ["Harry Potter"]
        ],

        inputs=movie_input
    )


    # -------------------------
    # BUTTON FUNCTION
    # -------------------------

    analyze_button.click(
        fn=analyze_movie,
        inputs=movie_input,
        outputs=output
    )


    # -------------------------
    # FOOTER
    # -------------------------

    gr.HTML("""
    <div class="footer">

        🎬 Powered by LangChain • Groq • Llama AI

        <br><br>

        Made with ❤️ using Generative AI

    </div>
    """)


# -------------------------
# RUN APP
# -------------------------

demo.launch()

C:\Users\nandi\AppData\Local\Temp\ipykernel_11116\725944872.py:63: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


In [21]:
def analyze_movie(movie_name):

    try:

        if not movie_name.strip():
            return (
                "Please enter a movie name.",
                "", "", "", "", "", "", ""
            )

        # CHAIN 1
        movie_analysis = movie_chain.invoke({
            "movie_name": movie_name
        })

        # CHAIN 2
        recommendations = recommendation_chain.invoke({
            "movie_analysis": json.dumps(movie_analysis)
        })

        # Extract movie details

        title = movie_analysis.get("title", "Not Available")

        genre = ", ".join(
            movie_analysis.get("genre", [])
        )

        year = str(
            movie_analysis.get("year", "Not Available")
        )

        themes = ", ".join(
            movie_analysis.get("themes", [])
        )

        mood = movie_analysis.get(
            "mood",
            "Not Available"
        )

        summary = movie_analysis.get(
            "summary",
            "Not Available"
        )

        viewer_type = recommendations.get(
            "viewer_type",
            "Not Available"
        )

        # Recommendations

        rec_list = recommendations.get(
            "recommendations",
            []
        )

        recommendation_text = ""

        for i, movie in enumerate(rec_list, 1):

            recommendation_text += f"""
### 🍿 {i}. {movie.get('title')}

{movie.get('reason')}

---

"""

        return (
            title,
            genre,
            year,
            themes,
            mood,
            summary,
            viewer_type,
            recommendation_text
        )

    except Exception as e:

        return (
            "Error",
            "", "", "", "",
            str(e),
            "", ""
        )

In [23]:
with gr.Blocks(
    css=custom_css,
    theme=gr.themes.Base()
) as demo:


    # -------------------------
    # HEADER
    # -------------------------

    gr.HTML("""

    <div class="main-title">

    🎬 Movie AI Assistant

    </div>


    <div class="subtitle">

    Discover • Analyze • Recommend 🍿

    </div>

    """)


    # -------------------------
    # INPUT
    # -------------------------

    with gr.Row():

        with gr.Column():

            movie_input = gr.Textbox(
                label="🎥 Enter Movie Name",
                placeholder="Example: Interstellar"
            )

            analyze_button = gr.Button(
                "✨ Analyze Movie",
                variant="primary",
                elem_classes="analyze-btn"
            )


    # -------------------------
    # MOVIE TITLE
    # -------------------------

    gr.Markdown("# 🎬 Movie Analysis")


    title_output = gr.Textbox(
        label="🎬 Movie Title"
    )


    # -------------------------
    # MOVIE DETAILS
    # -------------------------

    with gr.Row():

        genre_output = gr.Textbox(
            label="🎭 Genre"
        )


        year_output = gr.Textbox(
            label="📅 Release Year"
        )


        mood_output = gr.Textbox(
            label="😊 Mood"
        )


    # -------------------------
    # THEMES
    # -------------------------

    themes_output = gr.Textbox(
        label="🧠 Themes"
    )


    # -------------------------
    # SUMMARY
    # -------------------------

    summary_output = gr.Textbox(
        label="📝 Movie Summary",
        lines=4
    )


    # -------------------------
    # VIEWER TYPE
    # -------------------------

    viewer_output = gr.Textbox(
        label="👤 Perfect For"
    )


    # -------------------------
    # RECOMMENDATIONS
    # -------------------------

    recommendation_output = gr.Markdown()


    # -------------------------
    # BUTTON ACTION
    # -------------------------

    analyze_button.click(

        fn=analyze_movie,

        inputs=movie_input,

        outputs=[

            title_output,

            genre_output,

            year_output,

            themes_output,

            mood_output,

            summary_output,

            viewer_output,

            recommendation_output

        ]

    )


    # -------------------------
    # EXAMPLES
    # -------------------------

    gr.Markdown("## 🎞️ Try These Movies")


    gr.Examples(

        examples=[

            ["Interstellar"],

            ["Inception"],

            ["Avatar"],

            ["Titanic"],

            ["The Dark Knight"],

            ["Harry Potter"]

        ],

        inputs=movie_input

    )


    # -------------------------
    # FOOTER
    # -------------------------

    gr.HTML("""

    <div class="footer">

    🎬 Powered by LangChain • Groq • Llama AI

    <br><br>

    Made with ❤️ using Generative AI

    </div>

    """)


# -------------------------
# RUN APP
# -------------------------

demo.launch()

C:\Users\nandi\AppData\Local\Temp\ipykernel_11116\3552120716.py:1: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


In [24]:
custom_css = """

/* -------------------------
   BACKGROUND
------------------------- */

body {
    background: linear-gradient(
        135deg,
        #1a1026,
        #2b1648,
        #120b1f
    );
}

.gradio-container {
    background: transparent !important;
    color: white !important;
}


/* -------------------------
   MAIN TITLE
------------------------- */

.main-title {
    text-align: center;
    font-size: 48px;
    font-weight: bold;
    margin-top: 25px;

    background: linear-gradient(
        90deg,
        #ff4ecd,
        #a855f7,
        #6366f1
    );

    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
}


/* -------------------------
   SUBTITLE
------------------------- */

.subtitle {
    text-align: center;
    font-size: 18px;
    color: #d8b4fe;
    margin-bottom: 35px;
}


/* -------------------------
   CARDS
------------------------- */

.card {

    background: rgba(35, 20, 55, 0.8);

    border: 1px solid #7c3aed;

    border-radius: 20px;

    padding: 20px;

    box-shadow:
        0px 0px 20px rgba(168, 85, 247, 0.25);
}


/* -------------------------
   BUTTON
------------------------- */

.analyze-btn button {

    background: linear-gradient(
        90deg,
        #ec4899,
        #a855f7
    ) !important;

    color: white !important;

    border: none !important;

    height: 55px;

    font-size: 18px !important;

    font-weight: bold !important;

    border-radius: 15px !important;

    transition: 0.3s;
}


/* BUTTON HOVER */

.analyze-btn button:hover {

    transform: scale(1.03);

    box-shadow:
        0px 0px 25px
        rgba(236, 72, 153, 0.6);
}


/* -------------------------
   TEXTBOX
------------------------- */

textarea,
input {

    background: #21152e !important;

    color: white !important;

    border-radius: 12px !important;

    border: 1px solid #7c3aed !important;
}


/* -------------------------
   LABELS
------------------------- */

label {

    color: #e9d5ff !important;

    font-weight: bold !important;
}


/* -------------------------
   RECOMMENDATION AREA
------------------------- */

.prose {

    color: white !important;

    background: rgba(35, 20, 55, 0.6);

    padding: 20px;

    border-radius: 15px;
}


/* -------------------------
   FOOTER
------------------------- */

.footer {

    text-align: center;

    color: #c4b5fd;

    margin-top: 40px;

    font-size: 15px;
}

"""

In [25]:
# ==========================================
# 🎨 BEAUTIFUL PURPLE + PINK DARK GRADIO UI
# ==========================================

import gradio as gr


# -------------------------
# CUSTOM CSS
# -------------------------

custom_css = """

/* MAIN BACKGROUND */

body {
    background: linear-gradient(
        135deg,
        #120b1f,
        #2b1648,
        #1a1026
    );
}


/* GRADIO CONTAINER */

.gradio-container {

    background: transparent !important;

    color: white !important;

    max-width: 1200px !important;

}


/* MAIN TITLE */

.main-title {

    text-align: center;

    font-size: 48px;

    font-weight: bold;

    margin-top: 30px;

    background: linear-gradient(
        90deg,
        #ff4ecd,
        #a855f7,
        #6366f1
    );

    -webkit-background-clip: text;

    -webkit-text-fill-color: transparent;

}


/* SUBTITLE */

.subtitle {

    text-align: center;

    font-size: 18px;

    color: #d8b4fe;

    margin-bottom: 35px;

}


/* CARD STYLE */

.gr-group {

    background: rgba(30, 20, 45, 0.85) !important;

    border: 1px solid #7c3aed !important;

    border-radius: 20px !important;

    padding: 20px !important;

    box-shadow:
        0px 0px 20px
        rgba(168, 85, 247, 0.20);

}


/* BUTTON */

.analyze-btn {

    background: linear-gradient(
        90deg,
        #ec4899,
        #a855f7
    ) !important;

    color: white !important;

    border: none !important;

    height: 55px !important;

    font-size: 18px !important;

    font-weight: bold !important;

    border-radius: 15px !important;

}


/* BUTTON HOVER */

.analyze-btn:hover {

    transform: scale(1.02);

    box-shadow:
        0px 0px 25px
        rgba(236, 72, 153, 0.6);

}


/* INPUT BOX */

textarea,
input {

    background: #21152e !important;

    color: white !important;

    border-radius: 12px !important;

    border: 1px solid #7c3aed !important;

}


/* INPUT FOCUS */

textarea:focus,
input:focus {

    border: 1px solid #ec4899 !important;

    box-shadow:
        0px 0px 10px
        rgba(236, 72, 153, 0.5);

}


/* LABELS */

label {

    color: #e9d5ff !important;

    font-weight: bold !important;

}


/* MARKDOWN TEXT */

.prose {

    color: white !important;

}


/* HEADINGS */

h1,
h2,
h3 {

    color: #f5d0fe !important;

}


/* RECOMMENDATION CARD */

.recommendation-box {

    background: rgba(35, 20, 55, 0.8);

    border: 1px solid #a855f7;

    border-radius: 18px;

    padding: 20px;

    margin-top: 20px;

}


/* EXAMPLES */

.gr-examples {

    background: rgba(30, 20, 45, 0.7);

    border-radius: 15px;

    padding: 15px;

}


/* FOOTER */

.footer {

    text-align: center;

    color: #c4b5fd;

    margin-top: 40px;

    font-size: 15px;

    padding: 20px;

}

"""


# ==========================================
# CREATE GRADIO APP
# ==========================================


with gr.Blocks(

    css=custom_css,

    theme=gr.themes.Base()

) as demo:


    # ==========================================
    # HEADER
    # ==========================================


    gr.HTML("""

    <div class="main-title">

        🎬 Movie AI Assistant

    </div>


    <div class="subtitle">

        Discover • Analyze • Recommend 🍿

    </div>

    """)


    # ==========================================
    # MAIN SECTION
    # ==========================================


    with gr.Row():


        # ----------------------------------
        # LEFT SIDE - INPUT
        # ----------------------------------


        with gr.Column(scale=1):


            with gr.Group():


                gr.Markdown("""

                ## 🎥 Find Your Movie

                Enter a movie name and let AI
                analyze it for you! ✨

                """)


                movie_input = gr.Textbox(

                    label="🎬 Movie Name",

                    placeholder="Example: Interstellar",

                    lines=1

                )


                analyze_button = gr.Button(

                    "✨ Analyze Movie",

                    elem_classes="analyze-btn"

                )


                gr.Markdown("""

                ### 🍿 Popular Movies

                Try these:

                🎬 Interstellar

                🎬 Inception

                🎬 Avatar

                🎬 Titanic

                🎬 Harry Potter

                """)


        # ----------------------------------
        # RIGHT SIDE - MOVIE RESULTS
        # ----------------------------------


        with gr.Column(scale=2):


            with gr.Group():


                gr.Markdown(

                    "## 🤖 AI Movie Analysis"

                )


                # MOVIE TITLE

                title_output = gr.Textbox(

                    label="🎬 Movie Title"

                )


                # MOVIE DETAILS

                with gr.Row():


                    genre_output = gr.Textbox(

                        label="🎭 Genre"

                    )


                    year_output = gr.Textbox(

                        label="📅 Release Year"

                    )


                    mood_output = gr.Textbox(

                        label="😊 Mood"

                    )


                # THEMES

                themes_output = gr.Textbox(

                    label="🧠 Themes"

                )


                # SUMMARY

                summary_output = gr.Textbox(

                    label="📝 Movie Summary",

                    lines=5

                )


                # VIEWER TYPE

                viewer_output = gr.Textbox(

                    label="👤 Perfect For"

                )


                # RECOMMENDATIONS

                gr.Markdown(

                    "## 🍿 You May Also Like"

                )


                recommendation_output = gr.Markdown(

                    elem_classes="recommendation-box"

                )


    # ==========================================
    # EXAMPLE MOVIES
    # ==========================================


    gr.Markdown(

        "## 🎞️ Quick Movie Examples"

    )


    gr.Examples(

        examples=[

            ["Interstellar"],

            ["Inception"],

            ["Avatar"],

            ["Titanic"],

            ["The Dark Knight"],

            ["Harry Potter"]

        ],

        inputs=movie_input

    )


    # ==========================================
    # BUTTON CLICK FUNCTION
    # ==========================================


    analyze_button.click(

        fn=analyze_movie,

        inputs=movie_input,

        outputs=[

            title_output,

            genre_output,

            year_output,

            themes_output,

            mood_output,

            summary_output,

            viewer_output,

            recommendation_output

        ]

    )


    # ==========================================
    # FOOTER
    # ==========================================


    gr.HTML("""

    <div class="footer">

        🎬 Powered by LangChain • Groq • Llama AI

        <br><br>

        Made with ❤️ using Generative AI

    </div>

    """)


# ==========================================
# RUN APPLICATION
# ==========================================


demo.launch()

C:\Users\nandi\AppData\Local\Temp\ipykernel_11116\167426845.py:252: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


In [26]:
# ==========================================
# 🎨 ATTRACTIVE SKY BLUE GRADIO UI
# ==========================================

import gradio as gr


# -------------------------
# CUSTOM CSS
# -------------------------

custom_css = """

/* =========================================
   ☁️ MAIN SKY BLUE BACKGROUND
========================================= */

body {
    background: linear-gradient(
        135deg,
        #87CEEB,
        #B0E0E6,
        #E0F7FA
    ) !important;

    min-height: 100vh;
}


.gradio-container {

    background: transparent !important;

    max-width: 1250px !important;

}


/* =========================================
   🎬 MAIN TITLE
========================================= */

.main-title {

    text-align: center;

    font-size: 58px;

    font-weight: 900;

    color: #003B73;

    margin-top: 30px;

    letter-spacing: 2px;

    text-shadow:
        3px 3px 0px #FFFFFF,
        6px 6px 12px rgba(0, 70, 120, 0.35);

}


/* =========================================
   ✨ SUBTITLE
========================================= */

.subtitle {

    text-align: center;

    font-size: 22px;

    font-weight: bold;

    color: #075985;

    margin-bottom: 35px;

    text-shadow:
        1px 1px 3px rgba(255,255,255,0.8);

}


/* =========================================
   💎 GLASS CARDS
========================================= */

.gr-group {

    background:
        rgba(255, 255, 255, 0.55) !important;

    backdrop-filter:
        blur(12px);

    border:
        2px solid rgba(255,255,255,0.8) !important;

    border-radius: 25px !important;

    padding: 25px !important;

    box-shadow:
        0px 10px 30px rgba(0, 80, 130, 0.25);

}


/* =========================================
   📝 HEADINGS
========================================= */

h1,
h2,
h3 {

    color: #003B73 !important;

    font-size: 28px !important;

    font-weight: 800 !important;

    text-shadow:
        2px 2px 4px rgba(255,255,255,0.8);

}


/* =========================================
   🎬 INPUT LABEL
========================================= */

label {

    color: #003B73 !important;

    font-size: 18px !important;

    font-weight: bold !important;

}


/* =========================================
   📦 TEXTBOX
========================================= */

textarea,
input {

    background:
        rgba(255,255,255,0.85) !important;

    color: #003B73 !important;

    font-size: 17px !important;

    font-weight: 600 !important;

    border:
        2px solid #38BDF8 !important;

    border-radius: 15px !important;

}


/* =========================================
   🔍 INPUT FOCUS
========================================= */

textarea:focus,
input:focus {

    border:
        2px solid #0284C7 !important;

    box-shadow:
        0px 0px 15px rgba(14,165,233,0.5) !important;

}


/* =========================================
   🚀 ANALYZE BUTTON
========================================= */

.analyze-btn {

    background:
        linear-gradient(
            90deg,
            #0284C7,
            #38BDF8,
            #0EA5E9
        ) !important;

    color: white !important;

    font-size: 20px !important;

    font-weight: 800 !important;

    height: 60px !important;

    border: none !important;

    border-radius: 18px !important;

    box-shadow:
        0px 8px 20px rgba(2,132,199,0.4);

    transition:
        all 0.3s ease !important;

}


/* =========================================
   ✨ BUTTON HOVER
========================================= */

.analyze-btn:hover {

    transform:
        translateY(-3px) scale(1.02);

    box-shadow:
        0px 12px 30px rgba(2,132,199,0.6);

}


/* =========================================
   🍿 RECOMMENDATION BOX
========================================= */

.recommendation-box {

    background:
        linear-gradient(
            135deg,
            rgba(224,247,250,0.9),
            rgba(186,230,253,0.8)
        ) !important;

    border:
        2px solid #38BDF8 !important;

    border-radius: 20px !important;

    padding: 25px !important;

    box-shadow:
        0px 8px 25px rgba(14,165,233,0.25);

    color: #003B73 !important;

}


/* =========================================
   🎞️ EXAMPLES
========================================= */

.gr-examples {

    background:
        rgba(255,255,255,0.45) !important;

    border-radius:
        20px !important;

    padding:
        20px !important;

}


/* =========================================
   🌟 FOOTER
========================================= */

.footer {

    text-align: center;

    color: #075985;

    font-size: 17px;

    font-weight: bold;

    margin-top: 40px;

    padding: 25px;

    text-shadow:
        1px 1px 3px white;

}

"""


# ==========================================
# 🚀 CREATE GRADIO APP
# ==========================================

with gr.Blocks(

    css=custom_css,

    theme=gr.themes.Base()

) as demo:


    # ==========================================
    # 🎬 HEADER
    # ==========================================

    gr.HTML("""

    <div class="main-title">

        🎬 Movie AI Assistant

    </div>


    <div class="subtitle">

        Discover • Analyze • Recommend 🍿

    </div>

    """)


    # ==========================================
    # 📱 MAIN SECTION
    # ==========================================

    with gr.Row():


        # ======================================
        # 🎥 LEFT SIDE
        # ======================================

        with gr.Column(scale=1):

            with gr.Group():

                gr.Markdown("""

                ## 🎥 Find Your Movie

                Enter a movie name and let our AI
                analyze it for you! ✨

                """)


                movie_input = gr.Textbox(

                    label="🎬 Movie Name",

                    placeholder="Example: Interstellar",

                    lines=1

                )


                analyze_button = gr.Button(

                    "✨ Analyze Movie",

                    elem_classes="analyze-btn"

                )


                gr.Markdown("""

                ### 🍿 Popular Movies

                🎬 Interstellar

                🎬 Inception

                🎬 Avatar

                🎬 Titanic

                🎬 Harry Potter

                """)


        # ======================================
        # 🤖 RIGHT SIDE
        # ======================================

        with gr.Column(scale=2):

            with gr.Group():

                gr.Markdown(
                    "## 🤖 AI Movie Analysis"
                )


                title_output = gr.Textbox(
                    label="🎬 Movie Title"
                )


                with gr.Row():

                    genre_output = gr.Textbox(
                        label="🎭 Genre"
                    )


                    year_output = gr.Textbox(
                        label="📅 Release Year"
                    )


                    mood_output = gr.Textbox(
                        label="😊 Mood"
                    )


                themes_output = gr.Textbox(
                    label="🧠 Themes"
                )


                summary_output = gr.Textbox(

                    label="📝 Movie Summary",

                    lines=5

                )


                viewer_output = gr.Textbox(
                    label="👤 Perfect For"
                )


                gr.Markdown(
                    "## 🍿 You May Also Like"
                )


                recommendation_output = gr.Markdown(

                    elem_classes="recommendation-box"

                )


    # ==========================================
    # 🎞️ EXAMPLE MOVIES
    # ==========================================

    gr.Markdown(
        "## 🎞️ Quick Movie Examples"
    )


    gr.Examples(

        examples=[

            ["Interstellar"],

            ["Inception"],

            ["Avatar"],

            ["Titanic"],

            ["The Dark Knight"],

            ["Harry Potter"]

        ],

        inputs=movie_input

    )


    # ==========================================
    # 🔥 BUTTON CLICK
    # ==========================================

    analyze_button.click(

        fn=analyze_movie,

        inputs=movie_input,

        outputs=[

            title_output,

            genre_output,

            year_output,

            themes_output,

            mood_output,

            summary_output,

            viewer_output,

            recommendation_output

        ]

    )


    # ==========================================
    # 🌟 FOOTER
    # ==========================================

    gr.HTML("""

    <div class="footer">

        🎬 Powered by LangChain • Groq • Llama AI

        <br><br>

        ✨ Made with Generative AI ✨

    </div>

    """)


# ==========================================
# ▶️ RUN APPLICATION
# ==========================================

demo.launch()

C:\Users\nandi\AppData\Local\Temp\ipykernel_11116\2094672434.py:312: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


In [27]:
# ==========================================
# 🎬 MOVIE AI ASSISTANT
# COMPLETE NEW SKY BLUE UI
# ==========================================

import gradio as gr


# ==========================================
# 🎨 CUSTOM CSS
# ==========================================

custom_css = """

/* =========================================
   BACKGROUND
========================================= */

body {

    background:
    linear-gradient(
        135deg,
        #87CEEB,
        #B8E7FA,
        #EAF9FF
    ) !important;

}


/* MAIN CONTAINER */

.gradio-container {

    background: transparent !important;

    max-width: 1100px !important;

}


/* =========================================
   HEADER
========================================= */

.hero {

    text-align: center;

    padding: 45px 20px 30px 20px;

}


.hero-title {

    font-size: 64px;

    font-weight: 900;

    color: #003B73;

    letter-spacing: 2px;

    text-shadow:

        3px 3px 0px white,

        6px 6px 15px
        rgba(0,70,120,0.35);

}


.hero-subtitle {

    font-size: 22px;

    color: #075985;

    font-weight: bold;

    margin-top: 10px;

    text-shadow:
    1px 1px 3px white;

}


/* =========================================
   SEARCH CARD
========================================= */

.search-card {

    background:
    rgba(255,255,255,0.65);

    border-radius: 30px;

    padding: 35px;

    border:
    2px solid white;

    box-shadow:
    0px 12px 35px
    rgba(0,90,140,0.25);

}


/* =========================================
   SECTION TITLE
========================================= */

.section-title {

    text-align: center;

    font-size: 32px;

    font-weight: 900;

    color: #003B73;

    margin-top: 35px;

    text-shadow:
    2px 2px 5px white;

}


/* =========================================
   MOVIE TITLE
========================================= */

.movie-title textarea {

    font-size: 30px !important;

    font-weight: 900 !important;

    text-align: center;

    color: #003B73 !important;

}


/* =========================================
   INPUT
========================================= */

textarea,
input {

    background:
    rgba(255,255,255,0.9)
    !important;

    color:
    #003B73
    !important;

    font-size:
    18px
    !important;

    font-weight:
    600
    !important;

    border:
    2px solid
    #38BDF8
    !important;

    border-radius:
    18px
    !important;

}


/* =========================================
   BUTTON
========================================= */

.analyze-btn {

    background:
    linear-gradient(
        90deg,
        #0284C7,
        #0EA5E9,
        #38BDF8
    ) !important;

    color:
    white !important;

    font-size:
    21px !important;

    font-weight:
    800 !important;

    height:
    65px !important;

    border:
    none !important;

    border-radius:
    20px !important;

    box-shadow:

    0px 8px 25px
    rgba(2,132,199,0.4);

}


.analyze-btn:hover {

    transform:

    translateY(-4px);

    box-shadow:

    0px 15px 35px
    rgba(2,132,199,0.6);

}


/* =========================================
   INFO CARDS
========================================= */

.info-card {

    background:
    rgba(255,255,255,0.65);

    border:
    2px solid white;

    border-radius:
    25px;

    padding:
    20px;

    box-shadow:

    0px 8px 25px
    rgba(0,90,140,0.15);

}


/* =========================================
   LABELS
========================================= */

label {

    font-size:
    18px !important;

    font-weight:
    800 !important;

    color:
    #003B73 !important;

}


/* =========================================
   MARKDOWN
========================================= */

.prose {

    color:
    #003B73 !important;

    font-size:
    17px !important;

}


/* =========================================
   RECOMMENDATION
========================================= */

.recommendation-box {

    background:
    linear-gradient(
        135deg,
        #E0F7FF,
        #BCEBFA
    );

    border:
    2px solid
    #38BDF8;

    border-radius:
    25px;

    padding:
    30px;

    box-shadow:

    0px 10px 30px
    rgba(14,165,233,0.2);

}


/* =========================================
   FOOTER
========================================= */

.footer {

    text-align:
    center;

    font-size:
    17px;

    font-weight:
    bold;

    color:
    #075985;

    padding:
    40px;

}

"""


# ==========================================
# 🚀 CREATE APP
# ==========================================

with gr.Blocks(

    css=custom_css,

    theme=gr.themes.Base()

) as demo:


    # ======================================
    # 🎬 HERO HEADER
    # ======================================

    gr.HTML("""

    <div class="hero">

        <div class="hero-title">

            🎬 Movie AI Assistant

        </div>

        <div class="hero-subtitle">

            Your Personal AI Movie Intelligence 🍿

        </div>

    </div>

    """)


    # ======================================
    # 🔍 SEARCH SECTION
    # ======================================

    with gr.Group(
        elem_classes="search-card"
    ):


        gr.Markdown("""

        <div class="section-title">

            🔍 Search Your Movie

        </div>

        """)


        movie_input = gr.Textbox(

            label="🎬 Enter Movie Name",

            placeholder=
            "Try Interstellar, Avatar, Titanic...",

            lines=1

        )


        analyze_button = gr.Button(

            "✨ ANALYZE MOVIE",

            elem_classes="analyze-btn"

        )


    # ======================================
    # 🎬 MOVIE ANALYSIS
    # ======================================

    gr.Markdown("""

    <div class="section-title">

        🎬 Movie Analysis

    </div>

    """)


    # MOVIE TITLE

    with gr.Group(
        elem_classes="info-card"
    ):

        title_output = gr.Textbox(

            label="🎬 MOVIE TITLE",

            elem_classes="movie-title"

        )


    # ======================================
    # 📊 QUICK INFO CARDS
    # ======================================

    with gr.Row():


        with gr.Group(
            elem_classes="info-card"
        ):

            genre_output = gr.Textbox(

                label="🎭 GENRE"

            )


        with gr.Group(
            elem_classes="info-card"
        ):

            year_output = gr.Textbox(

                label="📅 RELEASE YEAR"

            )


        with gr.Group(
            elem_classes="info-card"
        ):

            mood_output = gr.Textbox(

                label="😊 MOOD"

            )


    # ======================================
    # 🧠 THEMES
    # ======================================

    with gr.Group(
        elem_classes="info-card"
    ):

        themes_output = gr.Textbox(

            label="🧠 MAIN THEMES"

        )


    # ======================================
    # 📝 SUMMARY
    # ======================================

    with gr.Group(
        elem_classes="info-card"
    ):

        summary_output = gr.Textbox(

            label="📝 STORY SUMMARY",

            lines=5

        )


    # ======================================
    # 👤 PERFECT VIEWER
    # ======================================

    with gr.Group(
        elem_classes="info-card"
    ):

        viewer_output = gr.Textbox(

            label="👤 PERFECT FOR"

        )


    # ======================================
    # 🍿 RECOMMENDATIONS
    # ======================================

    gr.Markdown("""

    <div class="section-title">

        🍿 Recommended For You

    </div>

    """)


    recommendation_output = gr.Markdown(

        elem_classes=
        "recommendation-box"

    )


    # ======================================
    # 🎞️ QUICK EXAMPLES
    # ======================================

    gr.Markdown("""

    <div class="section-title">

        🎞️ Try These Movies

    </div>

    """)


    gr.Examples(

        examples=[

            ["Interstellar"],

            ["Inception"],

            ["Avatar"],

            ["Titanic"],

            ["The Dark Knight"],

            ["Harry Potter"]

        ],

        inputs=movie_input

    )


    # ======================================
    # 🔥 BUTTON CLICK
    # ======================================

    analyze_button.click(

        fn=analyze_movie,

        inputs=movie_input,

        outputs=[

            title_output,

            genre_output,

            year_output,

            themes_output,

            mood_output,

            summary_output,

            viewer_output,

            recommendation_output

        ]

    )


    # ======================================
    # 🌟 FOOTER
    # ======================================

    gr.HTML("""

    <div class="footer">

        🎬 Powered by LangChain • Groq • Llama AI

        <br><br>

        ✨ Discover Your Next Favorite Movie ✨

    </div>

    """)


# ==========================================
# ▶️ RUN APP
# ==========================================

demo.launch()

C:\Users\nandi\AppData\Local\Temp\ipykernel_11116\3506568128.py:363: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.


In [28]:
# ==========================================
# 🎨 LUXURY NAVY + GOLD THEME
# ==========================================

custom_css = """

/* =========================================
   🌌 MAIN BACKGROUND
========================================= */

body {

    background:
    linear-gradient(
        135deg,
        #081C2C,
        #102A43,
        #1B3A4B
    ) !important;

    min-height: 100vh;

}


/* =========================================
   MAIN CONTAINER
========================================= */

.gradio-container {

    background: transparent !important;

    max-width: 1100px !important;

}


/* =========================================
   🎬 HERO SECTION
========================================= */

.hero {

    text-align: center;

    padding: 50px 20px 35px 20px;

}


.hero-title {

    font-size: 64px;

    font-weight: 900;

    color: #F4C95D;

    letter-spacing: 2px;

    text-shadow:

        3px 3px 0px #5C3D00,

        0px 0px 20px
        rgba(244,201,93,0.6);

}


.hero-subtitle {

    font-size: 22px;

    font-weight: bold;

    color: #FDF6E3;

    margin-top: 12px;

    text-shadow:

        1px 1px 5px
        rgba(0,0,0,0.5);

}


/* =========================================
   🔍 SEARCH CARD
========================================= */

.search-card {

    background:
    rgba(255, 248, 230, 0.95);

    border:

    2px solid #F4C95D !important;

    border-radius:

    30px !important;

    padding:

    35px !important;

    box-shadow:

    0px 12px 40px
    rgba(244,201,93,0.25);

}


/* =========================================
   📝 SECTION TITLE
========================================= */

.section-title {

    text-align: center;

    font-size: 32px;

    font-weight: 900;

    color: #F4C95D;

    margin:

    40px 0px 20px 0px;

    text-shadow:

    2px 2px 8px
    rgba(0,0,0,0.5);

}


/* =========================================
   💎 INFORMATION CARDS
========================================= */

.info-card {

    background:
    rgba(255, 248, 230, 0.96);

    border:

    1px solid #E0B84E !important;

    border-radius:

    25px !important;

    padding:

    22px !important;

    box-shadow:

    0px 8px 25px
    rgba(0,0,0,0.25);

}


/* =========================================
   🎬 MOVIE TITLE
========================================= */

.movie-title textarea {

    font-size:

    30px !important;

    font-weight:

    900 !important;

    text-align:

    center;

    color:

    #102A43 !important;

}


/* =========================================
   📦 INPUT BOX
========================================= */

textarea,
input {

    background:

    #FFFFFF !important;

    color:

    #102A43 !important;

    font-size:

    18px !important;

    font-weight:

    600 !important;

    border:

    2px solid #D4AF37 !important;

    border-radius:

    15px !important;

}


/* =========================================
   🏷️ LABELS
========================================= */

label {

    font-size:

    18px !important;

    font-weight:

    800 !important;

    color:

    #102A43 !important;

}


/* =========================================
   ✨ ANALYZE BUTTON
========================================= */

.analyze-btn {

    background:

    linear-gradient(
        90deg,
        #B8860B,
        #D4AF37,
        #F4C95D
    ) !important;

    color:

    #102A43 !important;

    font-size:

    21px !important;

    font-weight:

    900 !important;

    height:

    65px !important;

    border:

    none !important;

    border-radius:

    20px !important;

    box-shadow:

    0px 8px 25px
    rgba(212,175,55,0.45);

    transition:

    all 0.3s ease !important;

}


.analyze-btn:hover {

    transform:

    translateY(-4px);

    box-shadow:

    0px 15px 35px
    rgba(244,201,93,0.65);

}


/* =========================================
   🍿 RECOMMENDATION BOX
========================================= */

.recommendation-box {

    background:

    linear-gradient(
        135deg,
        #FFF8E6,
        #FDF1C9
    ) !important;

    color:

    #102A43 !important;

    border:

    2px solid #D4AF37 !important;

    border-radius:

    25px !important;

    padding:

    30px !important;

    box-shadow:

    0px 10px 30px
    rgba(212,175,55,0.25);

}


/* =========================================
   🌟 FOOTER
========================================= */

.footer {

    text-align:

    center;

    color:

    #F4C95D;

    font-size:

    17px;

    font-weight:

    bold;

    padding:

    40px;

    text-shadow:

    0px 0px 8px
    rgba(244,201,93,0.4);

}

"""

In [29]:
# ==========================================
# 🎬 MOVIE AI ASSISTANT
# LANGCHAIN + GROQ + GRADIO
# ==========================================


# ==========================================
# 📦 IMPORT LIBRARIES
# ==========================================

import os
import json
import gradio as gr

from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser


# ==========================================
# 🔑 API KEY
# ==========================================

os.environ["GROQ_API_KEY"] = "YOUR_GROQ_API_KEY"


# ==========================================
# 🤖 LLM MODEL
# ==========================================

llm = ChatGroq(

    model="llama-3.3-70b-versatile",

    temperature=0

)


# ==========================================
# 📄 JSON PARSER
# ==========================================

movie_parser = JsonOutputParser()

recommendation_parser = JsonOutputParser()


# ==========================================
# 🎬 CHAIN 1
# MOVIE ANALYZER
# ==========================================

movie_prompt = ChatPromptTemplate.from_template("""

You are an expert movie analyst.

Analyze the movie provided by the user.

Movie Name: {movie_name}

Return ONLY valid JSON.

Do not return markdown.

Do not write anything outside JSON.

Return this exact structure:

{{
    "title": "Movie Name",

    "genre": [
        "Genre 1",
        "Genre 2"
    ],

    "year": "Release Year",

    "themes": [
        "Theme 1",
        "Theme 2",
        "Theme 3"
    ],

    "mood": "Movie Mood",

    "summary": "Short movie summary"
}}

""")


movie_chain = (

    movie_prompt

    | llm

    | movie_parser

)


# ==========================================
# 🤖 CHAIN 2
# MOVIE RECOMMENDATION
# ==========================================

recommendation_prompt = ChatPromptTemplate.from_template("""

You are an expert movie recommendation system.

Analyze this movie information:

{movie_analysis}

Based on the movie information:

1. Identify the type of viewer who would enjoy this movie.

2. Recommend exactly 3 similar movies.

3. Give a short reason for every recommendation.

Return ONLY valid JSON.

Do not return markdown.

Do not write anything outside JSON.

Return this exact structure:

{{
    "viewer_type": "Type of Viewer",

    "recommendations": [

        {{
            "title": "Movie Name",

            "reason": "Why this movie is recommended"
        }},

        {{
            "title": "Movie Name",

            "reason": "Why this movie is recommended"
        }},

        {{
            "title": "Movie Name",

            "reason": "Why this movie is recommended"
        }}

    ]

}}

""")


recommendation_chain = (

    recommendation_prompt

    | llm

    | recommendation_parser

)


# ==========================================
# 🧠 MAIN FUNCTION
# ==========================================

def analyze_movie(movie_name):


    try:


        # ----------------------------------
        # CHECK EMPTY INPUT
        # ----------------------------------

        if not movie_name.strip():

            return (

                "Please enter a movie name.",

                "",

                "",

                "",

                "",

                "",

                "",

                ""

            )


        # ----------------------------------
        # 🎬 RUN CHAIN 1
        # ----------------------------------

        movie_analysis = movie_chain.invoke(

            {

                "movie_name": movie_name

            }

        )


        # ----------------------------------
        # 🤖 RUN CHAIN 2
        # ----------------------------------

        recommendation_result = recommendation_chain.invoke(

            {

                "movie_analysis": json.dumps(

                    movie_analysis,

                    indent=2

                )

            }

        )


        # ----------------------------------
        # 🎬 EXTRACT MOVIE DETAILS
        # ----------------------------------

        title = movie_analysis.get(

            "title",

            "Not Available"

        )


        genre = ", ".join(

            movie_analysis.get(

                "genre",

                []

            )

        )


        year = str(

            movie_analysis.get(

                "year",

                "Not Available"

            )

        )


        themes = ", ".join(

            movie_analysis.get(

                "themes",

                []

            )

        )


        mood = movie_analysis.get(

            "mood",

            "Not Available"

        )


        summary = movie_analysis.get(

            "summary",

            "Not Available"

        )


        viewer_type = recommendation_result.get(

            "viewer_type",

            "Not Available"

        )


        # ----------------------------------
        # 🍿 RECOMMENDATIONS
        # ----------------------------------

        recommendations = recommendation_result.get(

            "recommendations",

            []

        )


        recommendation_text = ""


        for i, movie in enumerate(

            recommendations,

            start=1

        ):


            movie_title = movie.get(

                "title",

                "Unknown Movie"

            )


            reason = movie.get(

                "reason",

                "No reason available"

            )


            recommendation_text += f"""

### 🎬 {i}. {movie_title}

{reason}

---

"""


        # ----------------------------------
        # 📤 RETURN OUTPUT
        # ----------------------------------

        return (

            title,

            genre,

            year,

            themes,

            mood,

            summary,

            viewer_type,

            recommendation_text

        )


    except Exception as e:


        return (

            "⚠️ Error",

            "",

            "",

            "",

            "",

            str(e),

            "",

            ""

        )


# ==========================================
# 🎨 CUSTOM CSS
# NAVY + GOLD LUXURY THEME
# ==========================================

custom_css = """

/* =========================================
   🌌 MAIN BACKGROUND
========================================= */

body {

    background:
    linear-gradient(
        135deg,
        #081C2C,
        #102A43,
        #1B3A4B
    ) !important;

    min-height: 100vh;

}


/* =========================================
   MAIN CONTAINER
========================================= */

.gradio-container {

    background: transparent !important;

    max-width: 1100px !important;

}


/* =========================================
   🎬 HERO SECTION
========================================= */

.hero {

    text-align: center;

    padding: 50px 20px 35px 20px;

}


.hero-title {

    font-size: 64px;

    font-weight: 900;

    color: #F4C95D;

    letter-spacing: 2px;

    text-shadow:

        3px 3px 0px #5C3D00,

        0px 0px 20px
        rgba(244,201,93,0.6);

}


.hero-subtitle {

    font-size: 22px;

    font-weight: bold;

    color: #FDF6E3;

    margin-top: 12px;

    text-shadow:

        1px 1px 5px
        rgba(0,0,0,0.5);

}


/* =========================================
   🔍 SEARCH CARD
========================================= */

.search-card {

    background:
    rgba(255, 248, 230, 0.95) !important;

    border:
    2px solid #F4C95D !important;

    border-radius:
    30px !important;

    padding:
    35px !important;

    box-shadow:

    0px 12px 40px
    rgba(244,201,93,0.25);

}


/* =========================================
   📝 SECTION TITLE
========================================= */

.section-title {

    text-align: center;

    font-size: 32px;

    font-weight: 900;

    color: #F4C95D;

    margin:
    40px 0px 20px 0px;

    text-shadow:

    2px 2px 8px
    rgba(0,0,0,0.5);

}


/* =========================================
   💎 INFORMATION CARDS
========================================= */

.info-card {

    background:
    rgba(255, 248, 230, 0.96) !important;

    border:
    1px solid #E0B84E !important;

    border-radius:
    25px !important;

    padding:
    22px !important;

    box-shadow:

    0px 8px 25px
    rgba(0,0,0,0.25);

}


/* =========================================
   🎬 MOVIE TITLE
========================================= */

.movie-title textarea {

    font-size:
    30px !important;

    font-weight:
    900 !important;

    text-align:
    center;

    color:
    #102A43 !important;

}


/* =========================================
   📦 INPUT BOX
========================================= */

textarea,
input {

    background:
    #FFFFFF !important;

    color:
    #102A43 !important;

    font-size:
    18px !important;

    font-weight:
    600 !important;

    border:
    2px solid #D4AF37 !important;

    border-radius:
    15px !important;

}


/* =========================================
   🏷️ LABELS
========================================= */

label {

    font-size:
    18px !important;

    font-weight:
    800 !important;

    color:
    #102A43 !important;

}


/* =========================================
   ✨ ANALYZE BUTTON
========================================= */

.analyze-btn {

    background:
    linear-gradient(
        90deg,
        #B8860B,
        #D4AF37,
        #F4C95D
    ) !important;

    color:
    #102A43 !important;

    font-size:
    21px !important;

    font-weight:
    900 !important;

    height:
    65px !important;

    border:
    none !important;

    border-radius:
    20px !important;

    box-shadow:

    0px 8px 25px
    rgba(212,175,55,0.45);

    transition:
    all 0.3s ease !important;

}


.analyze-btn:hover {

    transform:
    translateY(-4px);

    box-shadow:

    0px 15px 35px
    rgba(244,201,93,0.65);

}


/* =========================================
   🍿 RECOMMENDATION BOX
========================================= */

.recommendation-box {

    background:
    linear-gradient(
        135deg,
        #FFF8E6,
        #FDF1C9
    ) !important;

    color:
    #102A43 !important;

    border:
    2px solid #D4AF37 !important;

    border-radius:
    25px !important;

    padding:
    30px !important;

    box-shadow:

    0px 10px 30px
    rgba(212,175,55,0.25);

}


/* =========================================
   🌟 FOOTER
========================================= */

.footer {

    text-align:
    center;

    color:
    #F4C95D;

    font-size:
    17px;

    font-weight:
    bold;

    padding:
    40px;

    text-shadow:

    0px 0px 8px
    rgba(244,201,93,0.4);

}

"""


# ==========================================
# 🚀 CREATE GRADIO APPLICATION
# ==========================================

with gr.Blocks(

    css=custom_css,

    theme=gr.themes.Base()

) as demo:


    # ======================================
    # 🎬 HERO HEADER
    # ======================================

    gr.HTML("""

    <div class="hero">

        <div class="hero-title">

            🎬 Movie AI Assistant

        </div>


        <div class="hero-subtitle">

            Your Personal AI Movie Intelligence 🍿

        </div>

    </div>

    """)


    # ======================================
    # 🔍 SEARCH SECTION
    # ======================================

    with gr.Group(

        elem_classes="search-card"

    ):


        gr.Markdown("""

        <div class="section-title">

            🔍 Search Your Movie

        </div>

        """)


        movie_input = gr.Textbox(

            label="🎬 Enter Movie Name",

            placeholder=
            "Try Interstellar, Avatar, Titanic...",

            lines=1

        )


        analyze_button = gr.Button(

            "✨ ANALYZE MOVIE",

            elem_classes="analyze-btn"

        )


    # ======================================
    # 🎬 MOVIE ANALYSIS
    # ======================================

    gr.Markdown("""

    <div class="section-title">

        🎬 Movie Analysis

    </div>

    """)


    # ======================================
    # 🎬 MOVIE TITLE
    # ======================================

    with gr.Group(

        elem_classes="info-card"

    ):

        title_output = gr.Textbox(

            label="🎬 MOVIE TITLE",

            elem_classes="movie-title"

        )


    # ======================================
    # 📊 QUICK DETAILS
    # ======================================

    with gr.Row():


        with gr.Group(

            elem_classes="info-card"

        ):

            genre_output = gr.Textbox(

                label="🎭 GENRE"

            )


        with gr.Group(

            elem_classes="info-card"

        ):

            year_output = gr.Textbox(

                label="📅 RELEASE YEAR"

            )


        with gr.Group(

            elem_classes="info-card"

        ):

            mood_output = gr.Textbox(

                label="😊 MOOD"

            )


    # ======================================
    # 🧠 THEMES
    # ======================================

    with gr.Group(

        elem_classes="info-card"

    ):

        themes_output = gr.Textbox(

            label="🧠 MAIN THEMES"

        )


    # ======================================
    # 📝 SUMMARY
    # ======================================

    with gr.Group(

        elem_classes="info-card"

    ):

        summary_output = gr.Textbox(

            label="📝 STORY SUMMARY",

            lines=5

        )


    # ======================================
    # 👤 PERFECT VIEWER
    # ======================================

    with gr.Group(

        elem_classes="info-card"

    ):

        viewer_output = gr.Textbox(

            label="👤 PERFECT FOR"

        )


    # ======================================
    # 🍿 RECOMMENDATIONS
    # ======================================

    gr.Markdown("""

    <div class="section-title">

        🍿 Recommended For You

    </div>

    """)


    recommendation_output = gr.Markdown(

        elem_classes=
        "recommendation-box"

    )


    # ======================================
    # 🎞️ MOVIE EXAMPLES
    # ======================================

    gr.Markdown("""

    <div class="section-title">

        🎞️ Try These Movies

    </div>

    """)


    gr.Examples(

        examples=[

            ["Interstellar"],

            ["Inception"],

            ["Avatar"],

            ["Titanic"],

            ["The Dark Knight"],

            ["Harry Potter"]

        ],

        inputs=movie_input

    )


    # ======================================
    # 🔥 BUTTON CLICK FUNCTION
    # ======================================

    analyze_button.click(

        fn=analyze_movie,

        inputs=movie_input,

        outputs=[

            title_output,

            genre_output,

            year_output,

            themes_output,

            mood_output,

            summary_output,

            viewer_output,

            recommendation_output

        ]

    )


    # ======================================
    # 🌟 FOOTER
    # ======================================

    gr.HTML("""

    <div class="footer">

        🎬 Powered by LangChain • Groq • Llama AI

        <br><br>

        ✨ Discover Your Next Favorite Movie ✨

    </div>

    """)


# ==========================================
# ▶️ RUN APPLICATION
# ==========================================

demo.launch()

C:\Users\nandi\AppData\Local\Temp\ipykernel_11116\2461864801.py:788: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.
